# Durable RAG Ingestion: Pipelines That Survive Partial Failures, Re-Runs and Backfills

Every other technique in RAG assumes your documents made it into the index. Chunking strategy,
hybrid retrieval, reranking, evaluation. All of it is downstream of an ingestion job that someone
wrote once as a script and has been running in production ever since.

This tutorial is about that script. Specifically about the day it dies at document 3,000 of 5,000.

You will build the naive version first, break it, and **count what the retry costs you**. Then you
will build the version that keeps its place.

Sections 1 to 5 run anywhere with no API key and no account. Sections 7 onward run the real
pipeline on [Inngest](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--durable-rag-ingestion-inngest--durable-rag-ingestion-tutorial&click=inngest-home&target=https%3A%2F%2Fwww.inngest.com%2F%3Futm_source%3Ddiamantai%26utm_medium%3Dgithub%26utm_campaign%3Ddurable-rag-ingestion&text=Inngest), a durable execution platform that records each step a
function completes so a retry can resume instead of starting over. Section 6 introduces it
properly for readers who have not used it. Running it locally is a dev server and one command.

## 1. A corpus and five stages

Ingestion is always the same five stages: fetch, parse, chunk, embed, upsert.

We need a corpus big enough that a partial failure hurts. 5,000 documents is small for production
and more than enough to make the point.

Two notes on the fake parts. `embed()` is a deterministic stand-in for a paid embedding call, so
this notebook is reproducible and costs nothing to read. Swapping in a real model is one function
body. The counters are the interesting part: they record how much billable work each run does.

In [ ]:
import hashlib, random

def make_corpus(n=5000, seed=7):
    rnd = random.Random(seed)
    topics = ["retrieval", "embeddings", "chunking", "reranking", "evaluation"]
    docs = []
    for i in range(n):
        t = rnd.choice(topics)
        body = " ".join(rnd.choice(
            ["latency","recall","index","vector","token","corpus","query","chunk"]
        ) for _ in range(60))
        docs.append({"id": f"doc-{i:05d}", "topic": t, "text": f"{t}. {body}"})
    return docs

docs = make_corpus()
print(f"{len(docs)} documents")
print(docs[0])

In [ ]:
class Counters:
    """Every unit of billable work this pipeline does."""
    def __init__(self): self.parsed = self.embedded = self.upserted = 0
    def __repr__(self):
        return f"parsed={self.parsed} embedded={self.embedded} upserted={self.upserted}"

def parse(doc, c):
    c.parsed += 1
    return doc["text"]

def chunk(text, size=200):
    return [text[i:i+size] for i in range(0, len(text), size)]

def embed(chunk_text, c):
    """Stand-in for a paid embedding call. Deterministic, so results are reproducible."""
    c.embedded += 1
    h = hashlib.sha256(chunk_text.encode()).digest()
    return [b / 255.0 for b in h[:16]]

def content_hash(text):
    return hashlib.sha256(text.encode()).hexdigest()[:16]

## 2. The index, in its naive form

The default vector store setup appends. You hand it a chunk and a vector, it stores a row. It has
no opinion about whether it has seen that chunk before, because you never gave it a key that would
let it tell.

`append_only=True` is that behaviour. We turn it off in section 5 and watch what changes.

In [ ]:
class Index:
    def __init__(self, append_only=True):
        self.rows = []       # naive: every upsert appends
        self.by_key = {}     # idempotent: keyed on (doc_id, chunk_no, content_hash)
        self.append_only = append_only

    def upsert(self, doc_id, chunk_no, text, vec, c):
        c.upserted += 1
        if self.append_only:
            self.rows.append((doc_id, chunk_no, text, vec))
        else:
            self.by_key[(doc_id, chunk_no, content_hash(text))] = (text, vec)

    def size(self):
        return len(self.rows) if self.append_only else len(self.by_key)

    def duplicates(self):
        """Rows whose (doc, chunk, content) triple already appeared earlier in the index."""
        if not self.append_only: return 0
        seen, dupes = set(), 0
        for doc_id, chunk_no, text, _ in self.rows:
            k = (doc_id, chunk_no, content_hash(text))
            if k in seen: dupes += 1
            else: seen.add(k)
        return dupes

In [ ]:
class Boom(Exception):
    """Stands in for the embedding provider returning 429, or a PDF that parses to garbage."""

def ingest_naive(docs, index, c, fail_at=None):
    for n, doc in enumerate(docs):
        if fail_at is not None and n == fail_at:
            raise Boom(f"embedding provider returned 429 at document {n}")
        text = parse(doc, c)
        for i, ch in enumerate(chunk(text)):
            index.upsert(doc["id"], i, ch, embed(ch, c), c)
    return n + 1

## 3. Run it. It works.

On the happy path there is nothing wrong with this pipeline. That is exactly why it survives code
review and ends up in production.

In [ ]:
index, c0 = Index(), Counters()
done = ingest_naive(docs, index, c0)
print(f"ingested {done} documents")
print(f"work done: {c0}")
print(f"index rows: {index.size()}, duplicates: {index.duplicates()}")

## 4. Now kill it, and count what the retry costs

This is the section the whole tutorial exists for.

A real ingestion run dies partway. Rate limit, OOM, a deploy, a malformed file. The naive recovery
is to run the script again, because the script has no memory of what it finished.

Run one dies at document 3,000. Run two starts at document 0.

In [ ]:
index, c1 = Index(), Counters()
try:
    ingest_naive(docs, index, c1, fail_at=3000)
except Boom as e:
    print(f"run 1 died: {e}")

print(f"  run 1 completed work : {c1}")
print(f"  index rows so far    : {index.size()}")

In [ ]:
# The naive retry: same script, from the top, into the same index.
c2 = Counters()
ingest_naive(docs, index, c2)

print(f"run 2 completed work : {c2}")
print(f"index rows           : {index.size()}")
print(f"duplicate rows       : {index.duplicates()}")
print()
billed = c1.embedded + c2.embedded
print(f"embeddings billed across both runs : {billed}")
print(f"embeddings actually needed         : {c2.embedded}")
print(f"wasted                             : {c1.embedded}  ({billed/c2.embedded - 1:.0%} overspend)")

### What just happened

Two numbers matter here, and neither of them is the failure itself.

**8,735 embedding calls were paid for twice.** Run one embedded 8,735 chunks before it died. Run
two embedded all 14,571 from scratch, because nothing recorded that the first 8,735 were already
done. On a real embedding provider that is a 60 percent overspend on this ingest, and it scales
with how late the failure lands. A run that dies at 95 percent wastes almost a full corpus.

**8,735 duplicate rows are now in your index**, competing in retrieval. This is the one that hurts
later, because it does not raise an error anywhere. Retrieval just gets quietly worse: the same
chunk comes back twice in the top-k, crowding out a different chunk that should have made it.

The failure was never the expensive part. The redo was.

## 5. Fix one: make the upsert idempotent

The duplicate problem is not really about failures. It is about the index having no way to
recognise a chunk it has already stored.

Key each entry on document ID, chunk number and a hash of the content. Re-ingesting an unchanged
document now overwrites in place. Re-ingesting a changed document replaces the chunks that changed
and leaves the rest.

This one is worth doing whether or not you adopt anything else in this tutorial.

In [ ]:
index2, d1 = Index(append_only=False), Counters()
try:
    ingest_naive(docs, index2, d1, fail_at=3000)
except Boom:
    pass

d2 = Counters()
ingest_naive(docs, index2, d2)

print(f"index entries  : {index2.size()}")
print(f"duplicate rows : {index2.duplicates()}")
print()
print(f"embeddings still billed twice: {d1.embedded}")

Duplicates are gone. The wasted embedding calls are not.

That is the limit of fixing this at the storage layer: the index can deduplicate what it is
handed, but by then you have already paid to parse and embed the document. To stop paying twice,
something has to remember which work finished, and it has to remember it across a process that
died.

## 6. Where the record of completed work should live

Three options, in the order most teams reach for them.

**Rerun the script.** No memory. Cheapest to write, and you just measured what it costs.

**A state table you maintain.** Write a row per document, check it before processing. This works,
and most production pipelines end up here. The gap is the window between writing the row and doing
the work: if the process dies in there, the row lies to you. Closing that honestly means two-phase
writes and a reconciliation job, which is a distributed systems problem you now own.

**Put the record in the execution runtime.** Each stage of the pipeline is a recorded step. The
runtime journals what completed, and a retry replays the journal rather than re-running the work.
Resumption stops being state you maintain and becomes a property of how the function executes.

[Inngest](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--durable-rag-ingestion-inngest--durable-rag-ingestion-tutorial&click=inngest-home&target=https%3A%2F%2Fwww.inngest.com%2F%3Futm_source%3Ddiamantai%26utm_medium%3Dgithub%26utm_campaign%3Ddurable-rag-ingestion&text=Inngest) is the third option, and the rest of this tutorial builds on it.

### What Inngest is, if you have not used it

Enough of the model to read the rest of this tutorial. Skip to section 7 if you already use it.

Inngest is a durable execution platform. The part that matters here is where the durability lives:
in your own codebase, as ordinary Python functions, rather than in a queue and a worker fleet you
operate. There is no broker to run and no idempotency layer of your own to keep in sync.

Three pieces, and that is the whole model.

**Events.** A named JSON payload, like `rag/document.received`. Your application sends them, or one
function sends them to trigger others.

**Functions.** An ordinary Python function, decorated so the platform knows what starts it and how
it should behave when things go wrong:

```python
@client.create_function(
    fn_id="ingest-document",
    trigger=inngest.TriggerEvent(event="rag/document.received"),
    retries=3,
)
def ingest_document(ctx: inngest.ContextSync) -> dict:
    ...
```

Your own app serves these over HTTP. With FastAPI that is one line,
`inngest.fast_api.serve(app, client, [ingest_document])`, which mounts `/api/inngest`.

**Steps.** Inside a function, `ctx.step.run(step_id, handler)` marks a unit of work worth
remembering. Each step is executed as a separate request into your app, and its return value is
persisted in Inngest's state store. On a later attempt the step's code does not run again: the SDK
looks the result up by step ID and injects it into the return value of `step.run`.

That last sentence is the entire tutorial. A completed `embed` step returns its vector from the
state store on a retry, so the embedding call inside it is billed once rather than once per
attempt. Section 4 measured what the alternative costs.

Two consequences worth knowing before you write any of it. Anything non-deterministic or billable,
an API call, a database query, a paid embedding, belongs *inside* a step rather than between
steps, because only what is inside a step gets recorded. And step IDs are how results are matched
across attempts, which makes a step ID part of your compatibility surface: rename one and it
replays as new work. Section 13 comes back to that as a real cost rather than a footnote.

Retries, `throttle` and `concurrency` are arguments on the decorator, not infrastructure you stand
up. Section 7 uses all three.

None of this needs an account or an API key to try. `npx inngest-cli@latest dev` runs a local dev
server with a UI at `http://127.0.0.1:8288`, and `is_production=False` points the SDK at it.
Section 8 is the two commands.

## 7. The same pipeline, as durable steps

Below is the real thing. Two changes to notice.

Every stage is wrapped in `ctx.step.run(step_id, handler)`. When a function run is retried, Inngest
replays each completed step from its journal instead of executing it again. The embedding calls
inside a completed step are not re-billed.

Each document gets its own function run, triggered by its own event. A document that fails is one
failed run out of 5,000, not a dead pipeline.

This code is in `app.py` next to this notebook. It runs.

In [ ]:
# Excerpt from app.py. Do not run this cell on its own, it needs the dev server. See section 8.
import datetime
import inngest

client = inngest.Inngest(app_id="rag-ingestion", is_production=False)

@client.create_function(
    fn_id="ingest-document",
    trigger=inngest.TriggerEvent(event="rag/document.received"),
    retries=3,
    # Stay under the embedding provider's rate limit.
    throttle=inngest.Throttle(limit=100, period=datetime.timedelta(minutes=1)),
    # A backfill for one tenant cannot starve live ingestion for the others.
    concurrency=[inngest.Concurrency(key="event.data.tenant_id", limit=5)],
)
def ingest_document(ctx: inngest.ContextSync) -> dict:
    doc_id = ctx.event.data["doc_id"]
    doc = CORPUS[doc_id]

    text   = ctx.step.run("parse", lambda: parse(doc, COUNTERS))
    chunks = ctx.step.run("chunk", lambda: chunk(text))

    for i, ch in enumerate(chunks):
        # Each embed is its own step, so a failure at chunk 40
        # does not re-bill chunks 0 through 39.
        vec = ctx.step.run(f"embed-{i}", lambda ch=ch: embed(ch, COUNTERS))
        ctx.step.run(f"upsert-{i}",
                     lambda i=i, ch=ch, vec=vec: INDEX.upsert(doc_id, i, ch, vec, COUNTERS))

    return {"doc_id": doc_id, "chunks": len(chunks), "index_size": INDEX.size()}

### The two flow-control arguments

`throttle` caps how fast the function runs, across every document. That is what keeps a 5,000
document backfill inside your embedding provider's rate limit instead of collecting 429s.

`concurrency` with a `key` partitions that limit. Keyed on tenant, a customer re-indexing their
whole history gets 5 concurrent runs and everyone else still gets theirs. Without the key, one
backfill starves live ingestion.

Both are configuration on the function. Neither is a queue you operate.

## 8. Run it for real

No account, no keys. `is_production=False` points the SDK at the local dev server.

```bash
pip install -r requirements.txt

# terminal 1
uvicorn app:app --reload --port 8000

# terminal 2
npx inngest-cli@latest dev -u http://127.0.0.1:8000/api/inngest
```

Open http://127.0.0.1:8288 for the dev UI, then send the trigger event from the next cell. Watch
the runs appear, and open one to see each step recorded individually.

In [ ]:
# With both servers up, kick off the whole corpus.
import urllib.request, json

req = urllib.request.Request(
    "http://127.0.0.1:8288/e/dev_key",
    data=json.dumps({"name": "rag/corpus.ingest", "data": {}}).encode(),
    headers={"Content-Type": "application/json"},
)
print(urllib.request.urlopen(req).read().decode())

In [ ]:
# Progress, straight from the running app.
import urllib.request, json
print(json.dumps(json.load(urllib.request.urlopen("http://127.0.0.1:8000/stats")), indent=2))

## 9. Fan-out, and why one bad file stops mattering

`ingest_corpus` does one thing: it lists the documents and sends one event per document. Each of
those events starts an independent run of `ingest_document`.

The failure story changes completely. A PDF that parses to garbage fails its own run, retries on
its own schedule, and eventually lands in the review queue. The other 4,999 documents never notice.
Compare that to the naive loop, where the same file ends the run.

Note the batched `send_event` calls in `app.py`. Sending 5,000 events in one step makes one very
large payload, so 500 at a time keeps each step small.

**Why events here, and not `step.invoke`.** Inngest can also call one function from another and
hand back its return value, with
[`step.invoke`](https://www.inngest.com/docs/guides/invoking-functions-directly?guide=python):

```python
result = ctx.step.invoke(
    "ingest-one",
    function=ingest_document,
    data={"doc_id": doc_id, "tenant_id": tenant},
)
```

That is the better tool in two cases: the caller actually needs the child's result, or the work
wants its own retry and concurrency settings instead of the caller's. Neither applies to a 5,000
document backfill. `invoke` waits for the child to finish, and if the child fails the step fails
and raises a `NonRetriableError` in the calling run. Waiting and shared fate are exactly the two
couplings this section exists to remove: one unparseable PDF would be back to taking the corpus
down with it. Events buy 5,000 runs that succeed, fail and retry on their own.

## 10. A human gate for the documents that should not be guessed

Some documents should not go into the index on a best guess. A scanned contract that OCR'd badly,
a file whose parser returned three characters.

`ctx.step.wait_for_event()` suspends the run, for up to seven days here, without holding a worker
open. The run is durable while it waits. When a reviewer approves the document, the run picks up
exactly where it stopped.

`if_exp` is what routes the approval to the one run waiting for that document.

`step.invoke` does not replace this one either, and it is worth being clear about why. It does
save you a send-and-wait pair when a run is waiting on *another Inngest function* and wants what
that function returns. Here the run is waiting on a person. The decision arrives as an event from
outside the system, on a timescale of days, and there is no child function whose return value
would carry it.

In [ ]:
# Excerpt from app.py.
if not chunks:
    ctx.step.send_event(
        "flag-for-review",
        inngest.Event(name="rag/document.needs_review", data={"doc_id": doc_id}),
    )
    approval = ctx.step.wait_for_event(
        "await-approval",
        event="rag/document.approved",
        if_exp=f"async.data.doc_id == '{doc_id}'",
        timeout=datetime.timedelta(days=7),
    )
    if approval is None:
        return {"doc_id": doc_id, "status": "rejected-or-timed-out"}

## 11. Replay: fix the parser, re-run one step

This is the payoff of recording steps individually.

A parser bug corrupts 40 documents. In the naive pipeline you re-run the corpus. Here, the dev UI
shows you which runs failed and which step failed inside them. You fix the parser, deploy, and
replay those runs. The `embed` steps that already succeeded replay from the journal rather than
re-billing, and only the work after the fix actually executes.

Open any run in the dev UI at http://127.0.0.1:8288 to see the per-step timeline this is built on.

## 12. Query the finished index

The pipeline exists to serve retrieval, so here is the other end of it. Nothing exotic: cosine
similarity over the vectors the durable pipeline wrote.

In [ ]:
import math

def cosine(a, b):
    dot = sum(x*y for x, y in zip(a, b))
    na, nb = math.sqrt(sum(x*x for x in a)), math.sqrt(sum(y*y for y in b))
    return dot / (na * nb + 1e-12)

def search(index, query, k=3):
    c = Counters()
    qv = embed(query, c)
    scored = [(cosine(qv, vec), doc_key, text)
              for doc_key, (text, vec) in index.by_key.items()]
    return sorted(scored, key=lambda r: -r[0])[:k]

for score, key, text in search(index2, "how do I measure recall"):
    print(f"{score:.3f}  {key[0]} chunk {key[1]}  {text[:60]}...")

## 13. When you do not need any of this

Worth saying plainly, because a tutorial that only sells you the tool is not much use.

If your corpus is a few hundred documents that reindex nightly, and a full rerun costs you a few
cents and four minutes, use the script. The idempotent upserts from section 5 are still worth it,
because duplicate chunks degrade retrieval quietly and the fix is ten lines. The durable runtime
is not.

Durable execution starts paying when at least one of these is true:

- A full re-run is expensive, in tokens or in hours
- Partial failure is the normal case rather than an incident, which is what large corpora and
  flaky third-party parsers produce
- You have multiple tenants and one of them re-indexing must not degrade the others
- Some documents genuinely need a human in the loop before they enter the index

It also is not free. You adopt a runtime, your functions get shaped around steps, and step IDs
become part of your compatibility surface: renaming a step changes what replays. That is a real
cost and you should weigh it against the state table in section 6.

The honest summary is that the second option and the third are both defensible. The first one, the
one almost everybody is actually running, is the one this tutorial is arguing against.

## What to take away

The retry is where an ingestion pipeline's design shows. Everyone builds the happy path carefully
and then leaves recovery to a rerun, and that rerun quietly costs a share of your embedding bill
and fills your index with duplicates that make retrieval worse without ever raising an error.

You measured both in section 4. 8,735 wasted embedding calls and 8,735 duplicate rows, from a
single failure at 60 percent through the corpus.

Fixing it has two halves. Key your upserts on content so the index can recognise what it already
has. Then put the record of completed work somewhere that survives the process dying, so a retry
resumes instead of restarting.

---

*This tutorial was sponsored by [Inngest](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--durable-rag-ingestion-inngest--durable-rag-ingestion-tutorial&click=inngest-home&target=https%3A%2F%2Fwww.inngest.com%2F%3Futm_source%3Ddiamantai%26utm_medium%3Dgithub%26utm_campaign%3Ddurable-rag-ingestion&text=Inngest), the durable execution platform used
throughout. The architecture, the tradeoffs, and section 13 on when not to use it are the author's
own. Every number above comes from code in this notebook, so you can check it.*

*[Inngest docs](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--durable-rag-ingestion-inngest--durable-rag-ingestion-tutorial&click=inngest-docs&target=https%3A%2F%2Fwww.inngest.com%2Fdocs%3Futm_source%3Ddiamantai%26utm_medium%3Dgithub%26utm_campaign%3Ddurable-rag-ingestion&text=Inngest%20docs)*

![](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--durable-rag-ingestion-inngest--durable-rag-ingestion-tutorial)